In [21]:
zipcode_max_df = spark.sql("""
    SELECT Calltype, Zipcode, count(*) as count
    FROM spark_db.sf_fire_calls
    where Calltype is not null
    GROUP BY Calltype, Zipcode
    ORDER BY count DESC
    LIMIT 3
""")
zipcode_max_df.show()

+----------------+-------+-----+
|        Calltype|Zipcode|count|
+----------------+-------+-----+
|Medical Incident|  94102|16130|
|Medical Incident|  94103|14775|
|Medical Incident|  94110| 9995|
+----------------+-------+-----+



In [22]:
# Read data from the table into a df
fire_df = spark.read.table("spark_db.sf_fire_calls")
fire_df.show(10)

+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------+--------------------------+----------------------+------------------+--------------------+--------------------+-------------+---------+
|CallNumber|UnitID|IncidentNumber|        CallType|  CallDate| WatchDate|CallFinalDisposition|      AvailableDtTm|             Address|City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|        Neighborhood|            Location|        RowID|    Delay|
+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-

In [23]:
# apply transformations
df_1 = fire_df.select("Calltype", "Zipcode")
df_2 = df_1.where("Calltype is not null")
df_3 = df_2.groupBy("Calltype", "Zipcode").count().orderBy("count", ascending = False).limit(3)

In [24]:
# apply action like show or head
df_3.show()

+----------------+-------+-----+
|        Calltype|Zipcode|count|
+----------------+-------+-----+
|Medical Incident|  94102|16130|
|Medical Incident|  94103|14775|
|Medical Incident|  94110| 9995|
+----------------+-------+-----+



In [25]:
# explain() shows the plan that was used during the spark optimization of the spark action
df_3.explain(mode = "extended")

== Parsed Logical Plan ==
GlobalLimit 3
+- LocalLimit 3
   +- Sort [count#626L DESC NULLS LAST], true
      +- Aggregate [Calltype#416, Zipcode#423], [Calltype#416, Zipcode#423, count(1) AS count#626L]
         +- Filter isnotnull(Calltype#416)
            +- Project [Calltype#416, Zipcode#423]
               +- SubqueryAlias spark_catalog.spark_db.sf_fire_calls
                  +- Relation spark_catalog.spark_db.sf_fire_calls[CallNumber#413,UnitID#414,IncidentNumber#415,CallType#416,CallDate#417,WatchDate#418,CallFinalDisposition#419,AvailableDtTm#420,Address#421,City#422,Zipcode#423,Battalion#424,StationArea#425,Box#426,OriginalPriority#427,Priority#428,FinalPriority#429,ALSUnit#430,CallTypeGroup#431,NumAlarms#432,UnitType#433,UnitSequenceInCallDispatch#434,FirePreventionDistrict#435,SupervisorDistrict#436,Neighborhood#437,... 3 more fields] parquet

== Analyzed Logical Plan ==
Calltype: string, Zipcode: string, count: bigint
GlobalLimit 3
+- LocalLimit 3
   +- Sort [count#626L DESC

In [ ]:
# some more questions on spark transformations

In [26]:
# how many distinct types of calls made to the fire dept?

fire_df_dist_calls = (
    fire_df.select("CallType")
        .distinct()
        .withColumnRenamed("CallType", "Distinct CallTypes")
)
fire_df_dist_calls.count()

30

In [27]:
# what were distinct types of calls made to the fire dept
fire_df_dist_calls.show()

+--------------------+
|  Distinct CallTypes|
+--------------------+
|Elevator / Escala...|
|         Marine Fire|
|  Aircraft Emergency|
|Confined Space / ...|
|      Administrative|
|              Alarms|
|Odor (Strange / U...|
|Citizen Assist / ...|
|              HazMat|
|Watercraft in Dis...|
|           Explosion|
|           Oil Spill|
|        Vehicle Fire|
|  Suspicious Package|
|Extrication / Ent...|
|               Other|
|        Outside Fire|
|   Traffic Collision|
|       Assist Police|
|Gas Leak (Natural...|
+--------------------+
only showing top 20 rows


In [28]:
# Find out all response for delayed times greater than 5 mins?

fire_df_delayed = fire_df.where("Delay > 5").select("CallNumber", "Delay").orderBy("Delay", ascending = False)
fire_df_delayed.show()

+----------+---------+
|CallNumber|    Delay|
+----------+---------+
| 171580719|  1844.55|
|   2920036|1370.2333|
| 150283120|949.11664|
| 170340732|   931.45|
| 140710080|751.93335|
|  43500002|   734.05|
| 111000065|    721.4|
|  30800133|628.61664|
|  73520060|535.93335|
| 180763074|491.26666|
|  71380021|    437.8|
|  91410046|424.58334|
| 140700278|   414.85|
| 180210451|406.63333|
| 161933110|    405.7|
|  43500002|386.83334|
|  82490156|385.58334|
| 111240328|359.11667|
| 180813728|340.48334|
|  81730010|340.11667|
+----------+---------+
only showing top 20 rows


In [29]:
# What were the most common call types?

common_calltypes = fire_df.where("CallType is not null").groupBy("CallType").count().orderBy("count", ascending = False)
common_calltypes.show()

+--------------------+------+
|            CallType| count|
+--------------------+------+
|    Medical Incident|113794|
|      Structure Fire| 23319|
|              Alarms| 19406|
|   Traffic Collision|  7013|
|Citizen Assist / ...|  2524|
|               Other|  2166|
|        Outside Fire|  2094|
|        Vehicle Fire|   854|
|Gas Leak (Natural...|   764|
|        Water Rescue|   755|
|Odor (Strange / U...|   490|
|   Electrical Hazard|   482|
|Elevator / Escala...|   453|
|Smoke Investigati...|   391|
|          Fuel Spill|   193|
|              HazMat|   124|
|Industrial Accidents|    94|
|           Explosion|    89|
|Train / Rail Inci...|    57|
|  Aircraft Emergency|    36|
+--------------------+------+
only showing top 20 rows


In [30]:
# What zip codes accounted for most common calls?
zip_calltype_df = fire_df.where("CallType is not null").select("Zipcode", "CallType").groupBy("Zipcode", "CallType").count().orderBy("count", ascending = False)
zip_calltype_df.show()

+-------+----------------+-----+
|Zipcode|        CallType|count|
+-------+----------------+-----+
|  94102|Medical Incident|16130|
|  94103|Medical Incident|14775|
|  94110|Medical Incident| 9995|
|  94109|Medical Incident| 9479|
|  94124|Medical Incident| 5885|
|  94112|Medical Incident| 5630|
|  94115|Medical Incident| 4785|
|  94122|Medical Incident| 4323|
|  94107|Medical Incident| 4284|
|  94133|Medical Incident| 3977|
|  94117|Medical Incident| 3522|
|  94134|Medical Incident| 3437|
|  94114|Medical Incident| 3225|
|  94118|Medical Incident| 3104|
|  94121|Medical Incident| 2953|
|  94116|Medical Incident| 2738|
|  94132|Medical Incident| 2594|
|  94110|  Structure Fire| 2267|
|  94105|Medical Incident| 2258|
|  94102|  Structure Fire| 2229|
+-------+----------------+-----+
only showing top 20 rows


In [31]:
# What San Francisco neighborhoods are in the zip codes 94102 and 94103
neighborhood_df = fire_df.where("Zipcode in (94102, 94103)").select("Neighborhood").distinct()
neighborhood_df.show()

+--------------------+
|        Neighborhood|
+--------------------+
|    Western Addition|
|         Mission Bay|
|        Hayes Valley|
|Financial Distric...|
|            Nob Hill|
|             Mission|
|          Tenderloin|
|        Potrero Hill|
| Castro/Upper Market|
|     South of Market|
+--------------------+



In [32]:
# What was the sum of all calls, average, min and max of the response times for calls?
sum_avg_all = fire_df.selectExpr("sum(NumAlarms)", "avg(Delay)", "min(Delay)", "max(Delay)")
sum_avg_all.show()

+--------------+------------------+-----------+----------+
|sum(NumAlarms)|        avg(Delay)| min(Delay)|max(Delay)|
+--------------+------------------+-----------+----------+
|        176170|3.8923641541750342|0.016666668|   1844.55|
+--------------+------------------+-----------+----------+



In [33]:
# How many distinct years of data is in the CSV file?
distint_years_df = fire_df.selectExpr("year(CallDate) as year").distinct()
distint_years_df.show()

+----+
|year|
+----+
|2003|
|2018|
|2015|
|2006|
|2013|
|2014|
|2012|
|2009|
|2016|
|2005|
|2010|
|2011|
|2008|
|2017|
|2002|
|2007|
|2004|
|2001|
|2000|
|NULL|
+----+



In [36]:
# What week of the year in 2018 had the most fire calls?
week_df = fire_df.selectExpr("weekofyear(CallDate) as week_year").where("year(CallDate) = 2018").groupBy("week_year").count().orderBy("count", ascending = False)
week_df.show()

+---------+-----+
|week_year|count|
+---------+-----+
|        1|  279|
|       27|  261|
|        6|  261|
|       49|  238|
|       23|  236|
|       14|  236|
|       10|  226|
|       40|  217|
|       36|  202|
|       18|  197|
|       45|  194|
|       31|  165|
|       19|  152|
|       32|  151|
|        2|  137|
|        9|  129|
|       44|  126|
|        5|  123|
|       15|  112|
|       41|   90|
+---------+-----+
only showing top 20 rows


In [41]:
# What neighborhoods in San Francisco had the worst response time in 2018?
worst_delay_df = fire_df.select("Neighborhood", "Delay").groupBy("Neighborhood").avg("Delay").withColumnRenamed("avg(Delay)", "avg_delay").orderBy("avg_delay", ascending = False)
worst_delay_df.show()

+--------------------+------------------+
|        Neighborhood|         avg_delay|
+--------------------+------------------+
|     Treasure Island| 5.471499992963633|
|            Presidio|  4.96537535665225|
|         Mission Bay| 4.530760579459458|
|        McLaren Park|4.3098228641062795|
|                None| 4.307180866893617|
|          Twin Peaks| 4.294008406036819|
|    Golden Gate Park| 4.249903661308284|
|           Lakeshore| 4.201812142693754|
|Bayview Hunters P...| 4.150424641382069|
|            Seacliff| 4.137820518730768|
|   Visitacion Valley| 4.104289813916257|
|           Glen Park| 4.081832549182464|
|           Chinatown| 4.033199320980968|
|     South of Market|  4.00429725713771|
|      Bernal Heights| 4.004084867082827|
|        Lincoln Park| 4.003714848554217|
|        Russian Hill| 3.974167896072483|
|          Noe Valley| 3.951182856510139|
|      Outer Richmond| 3.939052998490439|
|         North Beach|3.9345745601868467|
+--------------------+------------